#### Just starter -- actual project code is after this code cell

In [2]:
import os
from huggingface_hub import InferenceClient
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()

# Initialize the client with the inference provider
client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

# ── Step 1:
model_small = "sentence-transformers/all-MiniLM-L6-v2"
model_large = "sentence-transformers/all-mpnet-base-v2"


# ── Step 2: A small test set with known relationships ─────────────
sentences = [
"The university requires a CNIC copy for admission.",   # 0
"Students must submit a copy of their national ID card.",  # 1 -- paraphrase of 0
"The cafeteria is open from 8am to 6pm.", # 2 -- unrelated
"Admission documents include the CNIC and academic transcripts.", # 3 -- related to 0
]


# ── Step 3: Encode with both models (normalised for fair comparison) ──
emb_small = client.feature_extraction(text=sentences,model=model_small,normalize=True)
emb_large = client.feature_extraction(text=sentences,model=model_large,normalize=True)


print("Small model dims:", emb_small.shape)
print("Large model dims:", emb_large.shape)


# ── Step 4: Compare similarity for the same pairs across both models ──
def report(emb,name):
    sims = cosine_similarity(emb)
    print(f"----{name}----")
    print(f"Paraphrase pair (0 vs 1) : {sims[0][1]:.4f}")
    print(f"Unrelated pair (0 vs 2) : {sims[0][2]:.4f}")
    print(f"Related pair (0 vs 3) : {sims[0][3]:.4f}")


report(emb_small, "all-MiniLM-L6-v2")
report(emb_large, "all-mpnet-base-v2")


/home/madiha/My Notes/2026/Code/RAG (basic to advance)/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Small model dims: (4, 384)
Large model dims: (4, 768)
----all-MiniLM-L6-v2----
Paraphrase pair (0 vs 1) : 0.5093
Unrelated pair (0 vs 2) : 0.0627
Related pair (0 vs 3) : 0.7684
----all-mpnet-base-v2----
Paraphrase pair (0 vs 1) : 0.6209
Unrelated pair (0 vs 2) : 0.1147
Related pair (0 vs 3) : 0.8056


### Capstone start

In [2]:
import os
from huggingface_hub import InferenceClient
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()

# Initialize the client with the inference provider
client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

model_large = "sentence-transformers/all-mpnet-base-v2"


my_chunks = [
"students must complete Semester registration before the start of classes.However, students will be allowed to add and drop courses as per the academic calendar with thepermission of the concerned HoD..",  

"One credit hour means dir ect teaching a theory course for a mi ni mum of 15 academic hours per semester. In case of practical one credit hour means direct instructions as well as performance of the experiments in laboratory environment for a minimum duration of 45 academic hours. An academic hour at the University is of 50 minutes duration.", 

"Before the submission of PhD thesis, a scholar must have contributed as a sole author or as a co-author to at least one article, related to his research work presented in the dissertation, published or accepted for publication (with acceptance letter from the Editor / Sub-editor of the journal) in international ISI indexed journal and or acceptable to the HEC.",

"The academic standing of a student is considered excellent if he/ achieves a Semester-GPA >=3.67 and CGPA >=3.33 Good The academic performance of a student in a semester is considered good if his/ Semester GPA<3.67 & >=3.00A is less than",

"At the end of period of expulsion, the student may resume his studies. However, the resumption could only be made in the start of a regular semester. In all such cases, the concerned Head of Teaching Department shall have the authority to notify the students expelled from the University with copies to all concerned offices/authorities."

]
my_embeddings = client.feature_extraction(text=my_chunks,model=model_large,normalize=True)
print("Capstone embeddings shape:", my_embeddings.shape)

Capstone embeddings shape: (5, 768)


In [3]:
# pip install langchain sentence-transformers nltk
from langchain_text_splitters import RecursiveCharacterTextSplitter
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import sent_tokenize
import numpy as np

document = """
Admission Policy
Students applying for the BS Computer Science program must submit
the following documents: CNIC copy, academic transcripts, and two
recommendation letters. Applications are reviewed within four weeks
of submission. Late applications are not accepted under any circumstances.
Fee Structure
The semester fee is PKR 85,000, payable in two installments. The
first installment is due before classes commence, and the second is
due at the midpoint of the semester. Late payment incurs a 5 percent
penalty after the due date.
Refund Policy
Students who withdraw within the first two weeks are eligible for a
full refund. Withdrawals between weeks three and six receive a 50
percent refund. No refunds are issued after week six, except for
documented medical emergencies reviewed by the Dean's office.
"""

# ■■ METHOD 1: Fixed-size chunking ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
def fixed_size_chunk(text, chunk_size=300, overlap=0):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start+chunk_size])
        start += chunk_size - overlap
    return chunks

chunks_fixed = fixed_size_chunk(document, chunk_size=300)

# ■■ METHOD 2: Recursive character splitting ■■■■■■■■■■■■■■■■■■■■■■■
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks_recursive = splitter.split_text(document)

# ■■ METHOD 3: Sentence-based chunking ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
sentences = sent_tokenize(document)

def sentence_chunk(sentences, chunk_size=300):
    chunks, current, length = [], [], 0
    for s in sentences:
        if length + len(s) > chunk_size and current:
            chunks.append(" ".join(current))
            current, length = [], 0
        current.append(s)
        length += len(s)
    if current:
        chunks.append(" ".join(current))
    return chunks

chunks_sentence = sentence_chunk(sentences, chunk_size=300)

# ■■ COMPARE: print all 3 methods side by side ■■■■■■■■■■■■■■■■■■■■■■
print("=" * 60)
print(f"FIXED-SIZE: {len(chunks_fixed)} chunks")
for i, c in enumerate(chunks_fixed):
    print(f"\n[{i}] {c[:80]}...")

print("=" * 60)
print(f"RECURSIVE: {len(chunks_recursive)} chunks")
for i, c in enumerate(chunks_recursive):
    print(f"\n[{i}] {c[:80]}...")

print("=" * 60)
print(f"SENTENCE-BASED: {len(chunks_sentence)} chunks")
for i, c in enumerate(chunks_sentence):
    print(f"\n[{i}] {c[:80]}...")

# ■■ QUALITY CHECK: does any chunk cut a sentence/word in half? ■■■■
def looks_cut_off(chunk):
    chunk = chunk.strip()
    return not chunk.endswith(('.', '!', '?')) if chunk else False

for name, chunks in [("Fixed", chunks_fixed),
                     ("Recursive", chunks_recursive),
                     ("Sentence", chunks_sentence)]:
    cut_count = sum(looks_cut_off(c) for c in chunks)
    print(f"{name}: {cut_count}/{len(chunks)} chunks appear cut mid-sentence")

# Expected finding: Fixed-size will have the MOST chunks ending
# mid-sentence. Recursive will have fewer. Sentence-based should have
# the FEWEST (ideally zero) -- write this finding down, it IS your
# Day 2 capstone log entry.

FIXED-SIZE: 3 chunks

[0] 
Admission Policy
Students applying for the BS Computer Science program must sub...

[1] ucture
The semester fee is PKR 85,000, payable in two installments. The
first in...

[2] are eligible for a
full refund. Withdrawals between weeks three and six receive ...
RECURSIVE: 3 chunks

[0] Admission Policy
Students applying for the BS Computer Science program must subm...

[1] Fee Structure
The semester fee is PKR 85,000, payable in two installments. The
f...

[2] Refund Policy
Students who withdraw within the first two weeks are eligible for ...
SENTENCE-BASED: 3 chunks

[0] 
Admission Policy
Students applying for the BS Computer Science program must sub...

[1] Fee Structure
The semester fee is PKR 85,000, payable in two installments. The
f...

[2] Refund Policy
Students who withdraw within the first two weeks are eligible for ...
Fixed: 2/3 chunks appear cut mid-sentence
Recursive: 1/3 chunks appear cut mid-sentence
Sentence: 0/3 chunks appear cut mid-sentence


## DAY 3

In [3]:
# Full capstone Day 3: chunk -> embed -> store -> retrieve with metadata
# pip install chromadb sentence-transformers langchain nltk pymupdf
import os  # Fixed: Moved to top to prevent NameError
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
import fitz
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

client_cdb = chromadb.PersistentClient(path="./capstone_db")
collection = client_cdb.get_or_create_collection(
    name="capstone_docs",
    metadata={"hnsw:space": "cosine"}
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
)

def ingest_pdf(pdf_path, doc_type="general"):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    chunks = splitter.split_text(full_text)
    
    # Generate embeddings and convert to list format for ChromaDB
    embeddings = client.feature_extraction(
        text=chunks, 
        model="BAAI/bge-small-en-v1.5",
    )
    # Ensure it's a list of lists of floats
    embeddings_list = embeddings.tolist() if hasattr(embeddings, 'tolist') else list(embeddings)
    
    filename = os.path.basename(pdf_path)
    ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [
        {"source": filename, "doc_type": doc_type, "chunk_index": i, "total_chunks": len(chunks)}
        for i in range(len(chunks))
    ]
    
    collection.add(
        ids=ids,
        embeddings=embeddings_list,
        documents=chunks,
        metadatas=metadatas
    )
    print(f"Ingested {len(chunks)} chunks from {filename}")

ingest_pdf("Final Revised Semester Rules 2019_260606_125443.pdf",doc_type="semester rules")

# # ■■ Demo with text if you have no PDFs yet ■■■■■■■■■■■■■■■■■■■
# demo_chunks = [
#     "Students must submit CNIC and transcripts for admission to KUST.",
#     "The semester fee is PKR 85,000 payable in two installments.",
#     "Full refund available for withdrawal in first two weeks.",
# ]


# demo_embs = client.feature_extraction(
#     text=demo_chunks,
#     model="BAAI/bge-small-en-v1.5"
# )
# demo_embs_list = demo_embs.tolist() if hasattr(demo_embs, 'tolist') else list(demo_embs)

# collection.add(
#     ids=["demo_0", "demo_1", "demo_2"],
#     embeddings=demo_embs_list,
#     documents=demo_chunks,
#     metadatas=[
#         {"source": "admission.pdf", "doc_type": "admission"},
#         {"source": "fees.pdf", "doc_type": "fees"},
#         {"source": "fees.pdf", "doc_type": "fees"},
#     ]
# )
print(f"Total chunks stored: {collection.count()}")

# ■■ Retrieval ■■■■■WW■■■■■■■■■■■WW■■■■■■■■■■■■■■■■■■■■■■■■■■■
def retrieve(query, doc_type_filter=None, top_k=3):
    q_emb = client.feature_extraction(
        text=[query],
        model="BAAI/bge-small-en-v1.5"
    )
    q_emb_list = q_emb.tolist() if hasattr(q_emb, 'tolist') else list(q_emb)
    
    where = {"doc_type": doc_type_filter} if doc_type_filter else None
    
    results = collection.query(
        query_embeddings=q_emb_list,  # Expects list of lists [[embs]]
        n_results=top_k,
        where=where
    )
    
    # Added safety check: Make sure results actually came back to prevent zip crashes
    if not results or not results["documents"] or not results["documents"][0]:
        return []
        
    return list(zip(results["documents"][0], results["metadatas"][0]))

print("\nQuery: what are the fees? (filter applied)")

for text, meta in retrieve("what are the fees", doc_type_filter="semester rules"):
    print(f"[{meta['source']}] {text[:500]}...")

print("\nQuery: admission requirements (no filter):")
for text, meta in retrieve("admission requirements"):
    print(f"[{meta['doc_type']}] {text[:70]}...")

Ingested 403 chunks from Final Revised Semester Rules 2019_260606_125443.pdf
Total chunks stored: 403

Query: what are the fees? (filter applied)
[Final Revised Semester Rules 2019_260606_125443.pdf] 23.5 
General Rules: ............................................................................................................................. 54 
24. 
Fee Structure ................................................................................................................................... 55...
[Final Revised Semester Rules 2019_260606_125443.pdf] i. 
In case of students already registered with KUST, registration fee and other fees will be 
charged as prescribed. However, the earlier deposited registration fee, if any, will be 
refunded/ adjusted. The original registration number will remain intact. 
ii. 
If a student wants to cancel his / her admission, then the refund of fee shall be made on the...
[Final Revised Semester Rules 2019_260606_125443.pdf] 24.1 Rules for the refun

## Day 4

In [6]:
# pip install chromadb sentence-transformers huggingface_hub
import chromadb
import json
import os
from sentence_transformers import SentenceTransformer

# ── Configuration ─────────────────────────────────────────────
from huggingface_hub import InferenceClient
client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
)

model = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_MODEL = model
LLM_MODEL = "openai/gpt-oss-120b"
TOP_K = 3
TEMPERATURE = 0.0
# ── Clients ───────────────────────────────────────────────────
embedder = SentenceTransformer(EMBED_MODEL)
chroma = chromadb.PersistentClient(path="./scratch_rag_db")
collection = chroma.get_or_create_collection(
name="kust_docs",
metadata={"hnsw:space": "cosine"}
)
# ── INGESTION ─────────────────────────────────────────────────
def ingest(chunks, sources):
    embeddings = embedder.encode(chunks, normalize_embeddings=True).tolist()  # client.feature_extraction()
    ids = [f"doc_{i}" for i in range(len(chunks))]
    metadatas = [{"source": s} for s in sources]
    collection.upsert(ids=ids, embeddings=embeddings,
    documents=chunks, metadatas=metadatas)
    print(f"Ingested {len(chunks)} chunks")
chunks = [
"Students must submit CNIC and transcripts for admission.",
"The semester fee is PKR 85,000 payable in two installments.",
"Late payment incurs a 5% penalty after the due date.",
"Full refund available within the first two weeks of semester.",
]
sources = ["admission.pdf", "fees.pdf", "fees.pdf", "fees.pdf"]
ingest(chunks, sources)
# ── RETRIEVAL ─────────────────────────────────────────────────
def retrieve(query, k=TOP_K):
    q_emb = embedder.encode([query], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=q_emb, n_results=k)
    retrieved = []
    for text, meta, score in zip(
    result["documents"][0],
    result["metadatas"][0],
    result["distances"][0]
    ):
         retrieved.append({
"text": text,
"source": meta["source"],
"score": 1 - score    # chroma returns distance, convert to similarity
})
    return retrieved
# ── PROMPT CONSTRUCTION ───────────────────────────────────────
SYSTEM_PROMPT = ("""You are a precise document assistant for KUST university.
Answer using ONLY the information in the provided context.
If the answer is not in the context, say: "I could not find this information."
Always cite the source document in parentheses after each relevant claim.""")
def build_prompt(retrieved_chunks, question):
  ctx = "\n\n".join([
f"[Source: {c['source']}]\n{c['text']}"
  for c in retrieved_chunks
])
  return f"Context:\n{ctx}\n\nQuestion: {question}\n\nAnswer:"
# ── GENERATION ────────────────────────────────────────────────
def generate(system_prompt, user_message):
  response = client.chat.completions.create(
    model=LLM_MODEL,
    temperature=TEMPERATURE,
    max_tokens=500,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
             ]
)
  return response.choices[0].message.content
# ── FULL RAG PIPELINE ─────────────────────────────────────────
def rag_query(question):
  print(f"\nQuery: {question}")
# Step 1: retrieve
  retrieved = retrieve(question, k=TOP_K)
  print(f"Retrieved {len(retrieved)} chunks:")
  for r in retrieved:
    print(f"[{r['score']:.3f}] {r['source']}: {r['text'][:50]}...")
# Step 2: build prompt
  user_msg = build_prompt(retrieved, question)
# Step 3: generate
  answer = generate(SYSTEM_PROMPT, user_msg)
  print(f"\nAnswer: {answer}")
# Step 4: return with sources
  return {
"answer":
answer,
"sources": [r["source"] for r in retrieved],
"chunks":retrieved
}
# ── RUN IT ────────────────────────────────────────────────────
# result = rag_query("What is the penalty for late fee payment?")
result = rag_query("What documents do I need for admission?")
#result = rag_query("Who is madiha?")
# NOT in context
# Last query should trigger the "I could not find this information" fallbackb

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4960.99it/s]


Ingested 4 chunks

Query: What documents do I need for admission?
Retrieved 3 chunks:
[0.474] admission.pdf: Students must submit CNIC and transcripts for admi...
[0.173] fees.pdf: The semester fee is PKR 85,000 payable in two inst...
[0.170] fees.pdf: Full refund available within the first two weeks o...

Answer: You need to submit your CNIC and your transcripts for admission (admission.pdf).


## Day 4 combined with day 3

In [1]:
# pip install chromadb langchain-text-splitters pymupdf sentence-transformers huggingface_hub
import os
import json
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient

# ── Configuration ─────────────────────────────────────────────
client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
)

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # Local model path
LLM_MODEL = "openai/gpt-oss-120b"
TOP_K = 3
TEMPERATURE = 0.0

# ── Clients ───────────────────────────────────────────────────
# Initialize your local embedding engine on the CPU
embedder = SentenceTransformer(EMBED_MODEL, device="cpu")

chroma = chromadb.PersistentClient(path="./scratch_rag_db")
collection = chroma.get_or_create_collection(
    name="kust_docs",
    metadata={"hnsw:space": "cosine"}
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
)

# ── INGESTION (LOCAL EMBEDDINGS) ──────────────────────────────
def ingest_pdf(pdf_path, doc_type="general"):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    chunks = splitter.split_text(full_text)
    
    # Changed: Running local vector calculations directly on your machine
    embeddings_list = embedder.encode(chunks, normalize_embeddings=True).tolist()
    
    filename = os.path.basename(pdf_path)
    ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [
        {"source": filename, "doc_type": doc_type, "chunk_index": i}
        for i in range(len(chunks))
    ]
    
    collection.upsert(
        ids=ids,
        embeddings=embeddings_list,
        documents=chunks,
        metadatas=metadatas
    )
    print(f"Ingested {len(chunks)} chunks from {filename}")

# Run Ingestion
ingest_pdf("Final Revised Semester Rules 2019_260606_125443.pdf", doc_type="semester rules")
print(f"Total chunks stored: {collection.count()}")

# ── RETRIEVAL (LOCAL EMBEDDINGS) ──────────────────────────────
def retrieve(query, doc_type_filter=None, k=TOP_K):
    # Changed: Generating query vector using your local embedder instance
    q_emb_list = embedder.encode([query], normalize_embeddings=True).tolist()
    
    where = {"doc_type": doc_type_filter} if doc_type_filter else None
    
    result = collection.query(
        query_embeddings=q_emb_list, 
        n_results=k,
        where=where
    )
    
    if not result or not result["documents"] or not result["documents"][0]:
        return []
        
    retrieved = []
    for text, meta, score in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ):
        retrieved.append({
            "text": text,
            "source": meta["source"],
            "score": 1 - score
        })
    return retrieved

# ── PROMPT CONSTRUCTION ───────────────────────────────────────
SYSTEM_PROMPT = ("""You are a precise document assistant for KUST university.
Answer using ONLY the information in the provided context.
If the answer is not in the context, say: "I could not find this information."
Always cite the source document in parentheses after each relevant claim.""")

def build_prompt(retrieved_chunks, question):
    ctx = "\n\n".join([
        f"[Source: {c['source']}]\n{c['text']}"
        for c in retrieved_chunks
    ])
    return f"Context:\n{ctx}\n\nQuestion: {question}\n\nAnswer:"

# ── GENERATION ────────────────────────────────────────────────
def generate(system_prompt, user_message):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        max_tokens=500,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    return response.choices[0].message.content

# ── FULL RAG PIPELINE ─────────────────────────────────────────
def rag_query(question, doc_type_filter=None):
    print(f"\nQuery: {question}")
    
    # Step 1: retrieve
    retrieved = retrieve(question, doc_type_filter=doc_type_filter, k=TOP_K)
    print(f"Retrieved {len(retrieved)} chunks:")
    for r in retrieved:
        print(f"[{r['score']:.3f}] {r['source']}: {r['text'][:50]}...")
        
    # Step 2: build prompt
    user_msg = build_prompt(retrieved, question)
    
    # Step 3: generate
    answer = generate(SYSTEM_PROMPT, user_msg)
    print(f"\nAnswer: {answer}")
    
    # Step 4: return structures
    return {
        "answer": answer,
        "sources": [r["source"] for r in retrieved],
        "chunks": retrieved
    }

# ── RUN IT ────────────────────────────────────────────────────
result = rag_query("What is the penalty for late fee payment?", doc_type_filter="semester rules")

/home/madiha/My Notes/2026/Code/RAG (basic to advance)/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4966.41it/s]


Ingested 403 chunks from Final Revised Semester Rules 2019_260606_125443.pdf
Total chunks stored: 403

Query: What is the penalty for late fee payment?
Retrieved 3 chunks:
[0.446] Final Revised Semester Rules 2019_260606_125443.pdf: respective Dean for approval.  
c. If granted appr...
[0.403] Final Revised Semester Rules 2019_260606_125443.pdf: the respective Head of Department for granting app...
[0.400] Final Revised Semester Rules 2019_260606_125443.pdf: i. 
In case of students already registered with KU...

Answer: The penalty is a **late‑registration fee that must be paid for each day the registration is late, at the approved daily rate**【Final Revised Semester Rules 2019_260606_125443.pdf】.
